# 08 - Table Export

**Project:** Causal ML for Climate Policy Effectiveness
**Purpose:** Generate the manuscript's tables from the regenerated outputs.

**Input:** `data/processed/analysis_panel.csv`, `data/processed/treatment_matrix.csv`,
`outputs/tables/att_gt_overall.csv`, `event_study_naive.csv`, `rambachan_roth_bounds.csv`,
`event_study_unit_trends.csv`
**Output:** `outputs/tables/table1_summary_stats.csv`, `table2_carbon_pricing_coverage.csv`,
`table3_naive_results.csv`, `table4_sensitivity_diagnostics.csv` (+ `.tex` versions)

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PROCESSED_DIR / "analysis_panel.csv")
treatment_df = pd.read_csv(PROCESSED_DIR / "treatment_matrix.csv")
print(f"Panel: {panel.shape}")

Panel: (6534, 68)


---
## Table 1: Summary Statistics

In [2]:
summary_vars = []
var_labels = {}

for v, l in [("co2", "CO2 Emissions (Mt)"), ("log_co2", "Log CO2 Emissions"),
             ("co2_per_capita", "CO2 per Capita (t)")]:
    if v in panel.columns:
        summary_vars.append(v)
        var_labels[v] = l

summary_vars.append("treat_carbon_price")
var_labels["treat_carbon_price"] = "Carbon Pricing (Treated)"

for v, l in [("gdp", "GDP"), ("population", "Population"),
             ("gdp_per_capita", "GDP per Capita"),
             ("urban_pct", "Urban Population (%)")]:
    if v in panel.columns:
        summary_vars.append(v)
        var_labels[v] = l

summary = panel[summary_vars].describe().T
summary.index = [var_labels.get(v, v) for v in summary.index]
summary = summary[["count", "mean", "std", "min", "50%", "max"]]
summary.columns = ["N", "Mean", "Std. Dev.", "Min", "Median", "Max"]

print("Table 1: Summary Statistics")
print("=" * 75)
print(summary.round(3).to_string())

latex = summary.round(3).to_latex(
    caption="Summary Statistics of Key Variables", label="tab:summary_stats", position="htbp",
)
with open(TABLE_DIR / "table1_summary_stats.tex", "w", encoding="utf-8") as f:
    f.write(latex)
summary.round(3).to_csv(TABLE_DIR / "table1_summary_stats.csv")
print()
print("Saved Table 1")

Table 1: Summary Statistics
                               N          Mean     Std. Dev.           Min        Median           Max
CO2 Emissions (Mt)        6422.0  1.491510e+02  6.877590e+02  0.000000e+00  9.648000e+00  1.171181e+04
Log CO2 Emissions         6422.0  2.256000e+00  2.586000e+00 -6.908000e+00  2.267000e+00  9.368000e+00
CO2 per Capita (t)        6422.0  4.672000e+00  7.848000e+00  0.000000e+00  2.338000e+00  3.647910e+02
Carbon Pricing (Treated)  6534.0  1.120000e-01  3.160000e-01  0.000000e+00  0.000000e+00  1.000000e+00
GDP                       5410.0  4.894066e+11  1.704830e+12  2.571720e+08  6.108450e+10  2.696602e+13
Population                6534.0  3.372726e+07  1.292951e+08  5.130000e+02  6.851751e+06  1.426437e+09
GDP per Capita            6317.0  1.257616e+04  1.978508e+04  1.667100e+02  4.264800e+03  2.143596e+05
Urban Population (%)      6468.0  5.560800e+01  2.367100e+01  5.275000e+00  5.618900e+01  1.000000e+02



Saved Table 1


---
## Table 2: Carbon-Pricing Coverage

From the World Carbon Pricing Database (Dolphin & Xiahou 2022) treatment
construction (Notebook 02): number of jurisdictions with a tax, an ETS, or
either, and the earliest/latest adoption years observed within the panel
window (1990-2022).

In [3]:
ever = treatment_df.groupby("iso3").agg(
    ever_tax=("any_tax", "max"), ever_ets=("any_ets", "max"), ever_treated=("treat_carbon_price", "max"),
)
# Instrument-specific adoption years (NOT the combined carbon-pricing
# adoption year) -- a country with an early tax and a later ETS must report
# each instrument's own earliest year, not reuse the tax year for the ETS row.
tax_adoption = treatment_df.loc[treatment_df["any_tax"] == 1].groupby("iso3")["year"].min()
ets_adoption = treatment_df.loc[treatment_df["any_ets"] == 1].groupby("iso3")["year"].min()
either_adoption = treatment_df.loc[treatment_df["treat_carbon_price"] == 1].groupby("iso3")["year"].min()

rows = []
for label, mask, adoption in [
    ("Carbon Tax", ever["ever_tax"] == 1, tax_adoption),
    ("Emissions Trading Scheme", ever["ever_ets"] == 1, ets_adoption),
    ("Either (Carbon Pricing)", ever["ever_treated"] == 1, either_adoption),
]:
    countries = ever.index[mask]
    years = adoption.reindex(countries).dropna()
    rows.append({
        "Instrument": label,
        "N Countries": int(mask.sum()),
        "Earliest Adoption": int(years.min()) if len(years) else None,
        "Latest Adoption": int(years.max()) if len(years) else None,
    })
coverage_table = pd.DataFrame(rows)

print("Table 2: Carbon-Pricing Coverage (World Carbon Pricing Database)")
print("=" * 70)
print(coverage_table.to_string(index=False))

latex = coverage_table.to_latex(
    index=False, caption="Carbon-Pricing Coverage by Instrument Type", label="tab:coverage",
)
with open(TABLE_DIR / "table2_carbon_pricing_coverage.tex", "w", encoding="utf-8") as f:
    f.write(latex)
coverage_table.to_csv(TABLE_DIR / "table2_carbon_pricing_coverage.csv", index=False)
print()
print("Saved Table 2")

Table 2: Carbon-Pricing Coverage (World Carbon Pricing Database)
              Instrument  N Countries  Earliest Adoption  Latest Adoption
              Carbon Tax           26               1990             2021
Emissions Trading Scheme           38               2005             2021
 Either (Carbon Pricing)           46               1990             2021

Saved Table 2


---
## Table 3: Naive Results

The headline-looking result before sensitivity diagnostics: Callaway-Santanna
overall ATT and the Sun-Abraham event-study estimates at selected horizons.

In [4]:
att_overall = pd.read_csv(TABLE_DIR / "att_gt_overall.csv")
event_naive = pd.read_csv(TABLE_DIR / "event_study_naive.csv")

rows = [{
    "Specification": "Callaway-Sant'Anna (overall ATT)",
    "Estimate": att_overall["estimate"].iloc[0],
    "Std. Error": att_overall["std_error"].iloc[0],
    "95% CI Lower": att_overall["ci_lower"].iloc[0],
    "95% CI Upper": att_overall["ci_upper"].iloc[0],
    "p-value": att_overall["p_value"].iloc[0],
}]
for et in [-5, -1, 0, 3, 5]:
    row = event_naive.loc[event_naive["event_time"] == et]
    if len(row):
        r = row.iloc[0]
        rows.append({
            "Specification": f"Sun-Abraham event time = {et:+d}",
            "Estimate": r["estimate"], "Std. Error": r["std_error"],
            "95% CI Lower": r["ci_lower"], "95% CI Upper": r["ci_upper"], "p-value": r["p_value"],
        })
naive_table = pd.DataFrame(rows)

print("Table 3: Naive Results (before sensitivity diagnostics)")
print("=" * 90)
print(naive_table.round(4).to_string(index=False))

latex = naive_table.round(4).to_latex(
    index=False, caption="Naive Carbon-Pricing Effect Estimates", label="tab:naive_results",
)
with open(TABLE_DIR / "table3_naive_results.tex", "w", encoding="utf-8") as f:
    f.write(latex)
naive_table.round(4).to_csv(TABLE_DIR / "table3_naive_results.csv", index=False)
print()
print("Saved Table 3")

Table 3: Naive Results (before sensitivity diagnostics)
                   Specification  Estimate  Std. Error  95% CI Lower  95% CI Upper  p-value
Callaway-Sant'Anna (overall ATT)   -0.2887      0.1061       -0.4966       -0.0807   0.0065
     Sun-Abraham event time = -5    0.2090      0.0385        0.1336        0.2844   0.0000
     Sun-Abraham event time = -1    0.0000      0.0000        0.0000        0.0000      NaN
     Sun-Abraham event time = +0   -0.0123      0.0058       -0.0237       -0.0009   0.0350
     Sun-Abraham event time = +3   -0.1023      0.0151       -0.1318       -0.0727   0.0000
     Sun-Abraham event time = +5   -0.4832      0.0397       -0.5609       -0.4054   0.0000

Saved Table 3


---
## Table 4: Sensitivity Diagnostics

The paper's central table: Rambachan-Roth relative-magnitude bounds (do the
naive estimates survive an honest accounting for the observed pre-trend?)
and the unit-specific-trends re-estimation (does the effect survive once
pre-existing divergent trajectories are absorbed?).

In [5]:
bounds = pd.read_csv(TABLE_DIR / "rambachan_roth_bounds.csv")
trends = pd.read_csv(TABLE_DIR / "event_study_unit_trends.csv")

print("Table 4a: Rambachan-Roth Relative-Magnitude Bounds")
print("=" * 70)
print(bounds.round(4).to_string(index=False))
print()
print("Table 4b: Event Study with Unit-Specific Linear Trends")
print("=" * 70)
print(trends.round(4).to_string(index=False))

latex_a = bounds.round(4).to_latex(
    index=False, caption="Rambachan-Roth (2023) Sensitivity Bounds", label="tab:rr_bounds",
)
with open(TABLE_DIR / "table4a_rambachan_roth_bounds.tex", "w", encoding="utf-8") as f:
    f.write(latex_a)
latex_b = trends.round(4).to_latex(
    index=False, caption="Event Study with Unit-Specific Linear Trends", label="tab:unit_trends",
)
with open(TABLE_DIR / "table4b_unit_trends.tex", "w", encoding="utf-8") as f:
    f.write(latex_b)

bounds.round(4).to_csv(TABLE_DIR / "table4a_rambachan_roth_bounds.csv", index=False)
trends.round(4).to_csv(TABLE_DIR / "table4b_unit_trends.csv", index=False)
print()
print("Saved Table 4a, 4b")

Table 4a: Rambachan-Roth Relative-Magnitude Bounds
     lb     ub method   Delta  Mbar                          target
-0.1147 0.0881   C-LF DeltaRM   0.5                 t=0 (immediate)
-0.1165 0.1165   C-LF DeltaRM   1.0                 t=0 (immediate)
-0.1165 0.1165   C-LF DeltaRM   1.5                 t=0 (immediate)
-0.1165 0.1165   C-LF DeltaRM   2.0                 t=0 (immediate)
-0.7932 0.0992   C-LF DeltaRM   0.5 t=5 (largest, last post-period)
-0.7932 0.6852   C-LF DeltaRM   1.0 t=5 (largest, last post-period)
-0.7932 0.7932   C-LF DeltaRM   1.5 t=5 (largest, last post-period)
-0.7932 0.7932   C-LF DeltaRM   2.0 t=5 (largest, last post-period)

Table 4b: Event Study with Unit-Specific Linear Trends
 event_time  estimate  std_error  ci_lower  ci_upper  p_value  n_cohorts
         -5   -0.0417     0.0260   -0.0927    0.0093   0.1092         16
         -4   -0.0246     0.0118   -0.0477   -0.0014   0.0374         16
         -3   -0.0094     0.0092   -0.0275    0.0087   0.3087 

In [6]:
print("=" * 60)
print("All manuscript tables generated")
print("=" * 60)
for f in sorted(TABLE_DIR.glob("table*.csv")):
    print(f"  {f.name}")

All manuscript tables generated
  table1_summary_stats.csv
  table2_carbon_pricing_coverage.csv
  table2_policy_coverage.csv
  table3_naive_results.csv
  table4a_rambachan_roth_bounds.csv
  table4b_unit_trends.csv
  table_sensitivity.csv
